In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
 
from pathlib import Path
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc as sk_auc


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
 
# GPU memory fix — prevents crash
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅  GPU active: {len(gpus)} GPU(s)")
else:
    print("⚠️  No GPU — go to Settings → Accelerator → GPU T4 x2")
 
# Mixed precision — half the memory, same accuracy
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("✅  Mixed precision enabled")
print(f"✅  TensorFlow {tf.__version__}")


In [ ]:
RAF_TRAIN_DIR = Path('/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/train')
RAF_TEST_DIR  = Path('/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test')
 
WORK_DIR  = Path('/kaggle/working')
CKPT_DIR  = WORK_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
 
IMG_SIZE  = 128    # smaller than 224 → faster training, less memory
                   # MobileNetV2 works great at 128×128
BATCH     = 32
EPOCHS    = 15     # enough to get good accuracy
STRESS    = 1
NO_STRESS = 0
 
# RAF-DB emotion → binary label mapping
# STRESS = Fear, Disgust, Sad, Anger (negative high-arousal emotions)
# NO STRESS = Happy, Surprise, Neutral
FOLDER_TO_LABEL = {
    '1': (NO_STRESS, 'Surprise'),
    '2': (STRESS,    'Fear'),
    '3': (STRESS,    'Disgust'),
    '4': (NO_STRESS, 'Happy'),
    '5': (STRESS,    'Sad'),
    '6': (STRESS,    'Anger'),
    '7': (NO_STRESS, 'Neutral'),
}
 
# Verify paths
print("\n📁  Path check:")
for name, path in [("Train", RAF_TRAIN_DIR), ("Test", RAF_TEST_DIR)]:
    ok = path.exists()
    print(f"  {'✅' if ok else '❌'}  {name}: {path}")


In [ ]:
print("  📊  EXPLORATORY DATA ANALYSIS — RAF-DB")
print("=" * 60)
 
# Build stats table
rows = []
for split, split_dir in [('Train', RAF_TRAIN_DIR), ('Test', RAF_TEST_DIR)]:
    for folder_num, (label, emotion) in FOLDER_TO_LABEL.items():
        folder = split_dir / folder_num
        if not folder.exists():
            continue
        count = len(list(folder.glob('*.jpg')) + list(folder.glob('*.png')))
        rows.append({
            'Split'   : split,
            'Folder'  : folder_num,
            'Emotion' : emotion,
            'Count'   : count,
            'Label'   : label,
            'Category': 'STRESS' if label == 1 else 'NO-STRESS',
        })
 
df = pd.DataFrame(rows)
 

In [ ]:
print("\n📋  .head():");        print(df.head(7))

In [ ]:
print(f"\n📐  .shape: {df.shape}")


In [ ]:
print(f"\n🔍  .isnull().sum():\n{df.isnull().sum()}")


In [ ]:
print(f"\n📊  .describe():\n{df.describe()}")


In [ ]:

print(f"\n⚖️   Class balance per split:")
print(df.groupby(['Split', 'Category'])['Count'].sum())

In [ ]:
print(f"\n📈  Per emotion (train only):")
train_df = df[df['Split'] == 'Train']
print(train_df[['Emotion', 'Count', 'Category']].to_string(index=False))


In [ ]:
print("""
🧠  WHY THESE EMOTIONS MAP TO STRESS:
  ├── Fear    → triggers fight-or-flight → cortisol spike → STRESS
  ├── Anger   → high physiological arousal → elevated heart rate → STRESS
  ├── Disgust → negative valence + arousal → stress response → STRESS
  ├── Sad     → prolonged negative affect → chronic stress → STRESS
  ├── Happy   → positive valence → no stress response → NO-STRESS
  ├── Surprise→ neutral/positive → no chronic stress → NO-STRESS
  └── Neutral → baseline → no stress → NO-STRESS
""")


In [ ]:
# Plot class balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RAF-DB Dataset Analysis', fontsize=14, fontweight='bold')
 
# Per emotion
colors = ['#e74c3c' if r['Label'] == 1 else '#2ecc71'
          for _, r in train_df.iterrows()]
bars = axes[0].bar(train_df['Emotion'], train_df['Count'],
                   color=colors, edgecolor='white')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20,
                 str(int(bar.get_height())),
                 ha='center', fontsize=9)
axes[0].set_xticklabels(train_df['Emotion'], rotation=30, ha='right')
axes[0].set_title('Images per Emotion\nRed=Stress | Green=No-Stress')
axes[0].grid(axis='y', alpha=0.3)
 
# Binary balance
binary = df.groupby('Category')['Count'].sum()
cols   = ['#e74c3c' if c == 'STRESS' else '#2ecc71' for c in binary.index]
bars2  = axes[1].bar(binary.index, binary.values, color=cols,
                     edgecolor='white', width=0.4)
total  = binary.sum()
for bar, val in zip(bars2, binary.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height()/2,
                 f'{val:,}\n({val/total*100:.1f}%)',
                 ha='center', va='center',
                 color='white', fontweight='bold', fontsize=13)
axes[1].set_title('Binary Class Balance\n(Class weights will fix imbalance)')
axes[1].grid(axis='y', alpha=0.3)
 
plt.tight_layout()
plt.savefig('/kaggle/working/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
 


In [ ]:
def load_split(split_dir, split_name):
    """
    Reads all images from the split directory.
    Returns numpy arrays: images (N,128,128,3) and labels (N,)
    """
    images = []
    labels = []
    failed = 0
 
    for folder_num, (label, emotion) in FOLDER_TO_LABEL.items():
        folder    = split_dir / folder_num
        img_paths = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
 
        for p in tqdm(img_paths,
                      desc=f"  {split_name} | {emotion:8s}",
                      ncols=75):
            img = cv2.imread(str(p))
            if img is None:
                failed += 1
                continue
 
            # BGR → RGB → resize → normalise
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img.astype(np.float32) / 255.0
 
            images.append(img)
            labels.append(label)
 
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
 
    print(f"\n  [{split_name}] Loaded: {len(images):,} | Failed: {failed}")
    print(f"  Stress    : {(labels==1).sum():,}")
    print(f"  No-Stress : {(labels==0).sum():,}")
    print(f"  Array size: {images.nbytes/1e9:.2f} GB")
    return images, labels
 
 
print("=" * 60)
print("  📂  LOADING IMAGES INTO RAM")
print("  ⏱️   Expected: ~3-5 minutes")
print("=" * 60)
 
X_train, y_train = load_split(RAF_TRAIN_DIR, 'Train')
X_test,  y_test  = load_split(RAF_TEST_DIR,  'Test')
 
print(f"\n  X_train shape: {X_train.shape}")
print(f"  X_test  shape: {X_test.shape}")
print(f"\n✅  All images loaded!")


In [ ]:
from sklearn.model_selection import train_test_split
 
# 15% of training data → validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size    = 0.15,
    random_state = SEED,
    stratify     = y_train   # preserve class ratio
)
 
print(f"  Train : {len(X_tr):,}")
print(f"  Val   : {len(X_val):,}")
print(f"  Test  : {len(X_test):,}")
 
# Class weights — fixes the 30/70 imbalance
cw  = compute_class_weight('balanced',
                            classes=np.array([0, 1]),
                            y=y_tr)
class_weight_dict = {0: float(cw[0]), 1: float(cw[1])}
print(f"\n  Class weights:")
print(f"    No-Stress (0): {cw[0]:.3f}")
print(f"    Stress    (1): {cw[1]:.3f}")
print(f"    → Model penalised {cw[1]/cw[0]:.1f}× more for missing stress")
 
 


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
 
backbone = MobileNetV2(
    input_shape = (IMG_SIZE, IMG_SIZE, 3),
    include_top = False,
    weights     = 'imagenet'
)
backbone.trainable = False   # freeze — Stage 1
 
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x      = backbone(inputs, training=False)
x      = layers.GlobalAveragePooling2D()(x)
x      = layers.Dropout(0.3)(x)
x      = layers.Dense(128, activation='relu')(x)
x      = layers.BatchNormalization()(x)
x      = layers.Dropout(0.2)(x)
# dtype='float32' needed for mixed precision stability
out    = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
 
model  = Model(inputs, out, name='StressDetector_MobileNetV2')
 
model.compile(
    optimizer = tf.keras.optimizers.Adam(1e-3),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)
 
model.summary()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
total     = model.count_params()
print(f"\n  Total params    : {total:,}")
print(f"  Trainable       : {trainable:,}")
print(f"  Frozen backbone : {total-trainable:,}")
print(f"\n✅  Model ready!")


In [ ]:
def augment(images, labels):
    """Light augmentation for training data."""
    images = tf.image.random_flip_left_right(images)
    images = tf.image.random_brightness(images, 0.15)
    images = tf.image.random_contrast(images, 0.85, 1.15)
    images = tf.clip_by_value(images, 0.0, 1.0)
    return images, labels
 
AUTOTUNE = tf.data.AUTOTUNE
 
# Training pipeline: shuffle → augment → batch → prefetch
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
    .shuffle(2000, seed=SEED)
    .batch(BATCH)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
 
# Val and test: NO augmentation — just batch
val_ds  = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH).prefetch(AUTOTUNE)
 
print(f"  Train batches : {len(train_ds)}")
print(f"  Val batches   : {len(val_ds)}")
print(f"  Test batches  : {len(test_ds)}")
print(f"✅  tf.data pipelines ready!")


In [ ]:
print("🚀  STAGE 1 — head only, backbone frozen")
history_s1 = model.fit(
    train_ds,
    epochs          = 5,
    validation_data = val_ds,
    class_weight    = class_weight_dict,
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(CKPT_DIR / 'best_s1.keras'),
            monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max',
            patience=3, restore_best_weights=True, verbose=1),
    ],
    verbose = 1,
)
print(f"✅  Stage 1 done! Best val AUC: {max(history_s1.history['val_auc']):.4f}")


In [ ]:
# Unfreeze top half of backbone
backbone.trainable = True
freeze_at = int(len(backbone.layers) * 0.5)
for i, layer in enumerate(backbone.layers):
    layer.trainable = (i >= freeze_at)
 
model.compile(
    optimizer = tf.keras.optimizers.Adam(1e-4),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)
 
print("🚀  STAGE 2 — fine-tuning top 50% of backbone")
history_s2 = model.fit(
    train_ds,
    epochs          = 10,
    validation_data = val_ds,
    class_weight    = class_weight_dict,
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(CKPT_DIR / 'best_final.keras'),
            monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max',
            patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, min_lr=1e-7, verbose=1),
    ],
    verbose = 1,
)
print(f"✅  Stage 2 done! Best val AUC: {max(history_s2.history['val_auc']):.4f}")


In [ ]:
hist   = {k: history_s1.history[k] + history_s2.history[k]
          for k in history_s1.history}
s1_len = len(history_s1.history['loss'])
xs     = range(1, len(hist['loss']) + 1)
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — Stage 1 + Stage 2', fontsize=14, fontweight='bold')
 
for ax, m, title in zip(axes,
    ['loss', 'accuracy', 'auc'],
    ['Loss', 'Accuracy', 'AUC (main metric)']):
    ax.plot(xs, hist[m],          color='#2E86C1', lw=2, label='Train')
    ax.plot(xs, hist[f'val_{m}'], color='#E74C3C', lw=2, label='Val')
    ax.axvline(s1_len + 0.5, color='gray', ls='--', lw=1.5, label='Stage 2')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Training curves saved!")


In [ ]:
#evaluation

model = tf.keras.models.load_model(str(CKPT_DIR / 'best_final.keras'))
 
probs = model.predict(test_ds, verbose=1).flatten()
preds = (probs >= 0.5).astype(int)
 
print("\n📋  Classification Report:")
print(classification_report(y_test, preds,
      target_names=['No-Stress', 'Stress'], digits=4))
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Test Set Evaluation', fontsize=14, fontweight='bold')
 
cm           = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()
lbl          = np.array([
    [f'TN\n{tn:,}\n({tn/len(y_test)*100:.1f}%)',
     f'FP\n{fp:,}\n({fp/len(y_test)*100:.1f}%)'],
    [f'FN ⚠️\n{fn:,}\n({fn/len(y_test)*100:.1f}%)',
     f'TP ✅\n{tp:,}\n({tp/len(y_test)*100:.1f}%)'],
])
sns.heatmap(cm, annot=lbl, fmt='', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No-Stress', 'Pred Stress'],
            yticklabels=['True No-Stress', 'True Stress'],
            linewidths=2, linecolor='white', annot_kws={'size': 12})
axes[0].set_title('Confusion Matrix', fontweight='bold')
 
fpr, tpr, _ = roc_curve(y_test, probs)
rauc        = sk_auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#E74C3C', lw=2.5, label=f'AUC = {rauc:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1.5, label='Random (0.50)')
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#E74C3C')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig('/kaggle/working/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
 
print(f"\n  🏆  AUC      : {rauc:.4f}")
print(f"      Accuracy : {(preds==y_test).mean():.4f}")
print(f"      Recall   : {tp/(tp+fn):.4f}")
print(f"      Precision: {tp/(tp+fp):.4f}")
 


In [ ]:
# Simple, visual, easy to explain to teacher:
# 1. Show what the model sees per emotion (confidence scores)
# 2. Show which emotions it finds most stressful
# 3. Show Grad-CAM heatmap (WHERE it looks on the face)
 
# ── Part A: Emotion-level stress scores ──────────────────────
print("Computing average stress score per emotion...")
 
emotion_scores = {}
for folder_num, (label, emotion) in FOLDER_TO_LABEL.items():
    folder    = RAF_TEST_DIR / folder_num
    img_paths = list(folder.glob('*.jpg'))[:50]   # 50 per emotion
 
    batch_imgs = []
    for p in img_paths:
        img = cv2.imread(str(p))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
        batch_imgs.append(img)
 
    if not batch_imgs: continue
    batch = np.array(batch_imgs)
    scores = model.predict(batch, verbose=0).flatten()
    emotion_scores[emotion] = {
        'mean_score'   : float(scores.mean()),
        'std_score'    : float(scores.std()),
        'true_label'   : label,
        'category'     : 'STRESS' if label == 1 else 'NO-STRESS',
    }
 
# Plot emotion-level stress scores
fig, ax = plt.subplots(figsize=(12, 6))
emotions_sorted = sorted(emotion_scores.keys(),
                          key=lambda e: emotion_scores[e]['mean_score'],
                          reverse=True)
scores_sorted   = [emotion_scores[e]['mean_score'] for e in emotions_sorted]
std_sorted      = [emotion_scores[e]['std_score']   for e in emotions_sorted]
colors_sorted   = ['#e74c3c' if emotion_scores[e]['true_label']==1
                   else '#2ecc71' for e in emotions_sorted]
 
bars = ax.barh(emotions_sorted, scores_sorted,
               color=colors_sorted, edgecolor='white',
               xerr=std_sorted, capsize=4)
 
# Add 0.5 threshold line
ax.axvline(x=0.5, color='black', ls='--', lw=2, label='Decision threshold (0.5)')
 
for bar, score in zip(bars, scores_sorted):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=10, fontweight='bold')
 
ax.set_xlabel('Average Stress Probability Score', fontsize=12)
ax.set_title('Model\'s Average Stress Score per Emotion\n'
             'Red = truly STRESS | Green = truly NO-STRESS | '
             'Dashed line = decision boundary',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, 1.1)
ax.grid(axis='x', alpha=0.3)
 
# Add labels on right side
for emotion in emotions_sorted:
    cat = emotion_scores[emotion]['category']
    idx = emotions_sorted.index(emotion)
    ax.text(1.05, idx, cat, va='center', fontsize=9,
            color='#e74c3c' if cat=='STRESS' else '#2ecc71',
            fontweight='bold')
 
plt.tight_layout()
plt.savefig('/kaggle/working/emotion_stress_scores.png',
            dpi=150, bbox_inches='tight')
plt.show()
 
print("\n📊  Emotion stress scores:")
for e in emotions_sorted:
    info = emotion_scores[e]
    print(f"  {e:10s}: {info['mean_score']:.3f}  [{info['category']}]")
 


In [ ]:
# ── Part B: Simple Grad-CAM (WHERE it looks) ─────────────────
print("\nGenerating Grad-CAM visualisations...")
 
def compute_gradcam(model, img_array):
    """Robust Grad-CAM for nested Keras 3 models (avoids InputLayer & Float16 errors)."""
    img_t = tf.cast(np.expand_dims(img_array, 0), tf.float32)

    # 1. Dynamically find the nested backbone model (MobileNetV2)
    backbone_idx = -1
    for i, layer in enumerate(model.layers):
        if hasattr(layer, 'layers'): 
            backbone_idx = i
            break
            
    if backbone_idx == -1:
        raise ValueError("Could not find the nested MobileNetV2 backbone.")
        
    backbone = model.layers[backbone_idx]
    head_layers = model.layers[backbone_idx + 1:]

    with tf.GradientTape() as tape:
        x = img_t
        
        # Pass through any initial layers, safely skipping the InputLayer
        for layer in model.layers[:backbone_idx]:
            if isinstance(layer, tf.keras.layers.InputLayer):
                continue
            x = layer(x)
            
        # 2. Get the 4D output from the backbone and tell tape to watch it
        conv_out = backbone(x, training=False)
        tape.watch(conv_out)
        
        # 3. Pass through the rest of the classification head
        x = conv_out
        for layer in head_layers:
            if isinstance(layer, (tf.keras.layers.Dropout, tf.keras.layers.BatchNormalization)):
                x = layer(x, training=False)
            else:
                x = layer(x)
                
        preds = x
        loss = preds[:, 0]

    # 4. Calculate gradients
    grads = tape.gradient(loss, conv_out)
    
    # Pool gradients and apply to feature maps
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis]).numpy()
    
    # Apply ReLU
    heatmap = np.maximum(heatmap, 0)
    
    # --- THE FIX: Cast to float32 so OpenCV doesn't crash ---
    heatmap = heatmap.astype(np.float32)
    
    # Normalize to 0-1
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
        
    return cv2.resize(heatmap, (img_array.shape[1], img_array.shape[0]))
 
# Show 2 stressed + 2 not-stressed with heatmaps
fig, axes = plt.subplots(2, 6, figsize=(20, 8))
fig.suptitle('Grad-CAM — WHERE the model looks to detect stress\n'
             'Red = high importance | Blue = ignored',
             fontsize=13, fontweight='bold')
 
examples = [
    ('2', 'Fear',    'STRESS'),
    ('6', 'Anger',   'STRESS'),
    ('4', 'Happy',   'NO-STRESS'),
    ('7', 'Neutral', 'NO-STRESS'),
]
 
for col, (folder_num, emotion, category) in enumerate(examples):
    folder    = RAF_TEST_DIR / folder_num
    img_paths = list(folder.glob('*.jpg'))
    if not img_paths: continue
 
    img_bgr = cv2.imread(str(img_paths[0]))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_arr = img_res.astype(np.float32) / 255.0
 
    score   = float(model.predict(np.expand_dims(img_arr, 0), verbose=0)[0][0])
    heatmap = compute_gradcam(model, img_arr)
 
    jet     = cv2.applyColorMap(np.uint8(255*heatmap), cv2.COLORMAP_JET)
    jet_rgb = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(jet_rgb, 0.45, img_res, 0.55, 0)
 
    color = '#e74c3c' if category == 'STRESS' else '#27ae60'
 
    # Original
    axes[0][col*1].imshow(img_res)
    axes[0][col*1].set_title(f'{emotion}\nScore: {score:.3f}',
                              fontsize=10, color=color, fontweight='bold')
    axes[0][col*1].axis('off')
 
    # Overlay
    axes[1][col*1].imshow(overlay)
    axes[1][col*1].set_title(f'Grad-CAM\n[{category}]',
                              fontsize=9, color=color)
    axes[1][col*1].axis('off')
 
    # Heatmap only
    axes[0][col*1 + 4] if col < 2 else None
 
axes[0][0].set_ylabel('Original', fontsize=11, rotation=0,
                        labelpad=60, va='center', fontweight='bold')
axes[1][0].set_ylabel('Grad-CAM\nOverlay', fontsize=11, rotation=0,
                        labelpad=60, va='center', fontweight='bold')
 
plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅  Explainability charts saved!")


In [ ]:
# Save to Kaggle output
MODEL_PATH = '/kaggle/working/stress_model_final.keras'
model.save(MODEL_PATH)
size = os.path.getsize(MODEL_PATH) / 1e6
print(f"✅  Model saved: {MODEL_PATH}  ({size:.1f} MB)")
print(f"    Download: right panel → Output tab")
 
# HuggingFace (paste your token to enable)
HF_TOKEN    = "YOUR_HF_TOKEN_HERE"
HF_USERNAME = "YOUR_HF_USERNAME"
HF_REPO     = "visual-stress-detection"
 
if "YOUR" not in HF_TOKEN:
    try:
        !pip install -q huggingface_hub
        from huggingface_hub import HfApi, create_repo
        api     = HfApi()
        repo_id = f"{HF_USERNAME}/{HF_REPO}"
        create_repo(repo_id, token=HF_TOKEN, repo_type="model", exist_ok=True)
        api.upload_file(path_or_fileobj=MODEL_PATH,
                        path_in_repo="stress_model_final.keras",
                        repo_id=repo_id, token=HF_TOKEN)
        for p in Path('/kaggle/working').glob('*.png'):
            api.upload_file(path_or_fileobj=str(p),
                            path_in_repo=f"plots/{p.name}",
                            repo_id=repo_id, token=HF_TOKEN)
        print(f"✅  Uploaded: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"❌  HF failed: {e}")
else:
    print("⏭️  HuggingFace skipped — paste token above to enable")
 
 
# ── Final predict function ────────────────────────────────────
def predict_stress(image_input):
    """
    Predicts stress from a face image.
    Input: file path (str) or BGR numpy array
    Output: dict with score, prediction, confidence
    """
    if isinstance(image_input, str):
        img = cv2.imread(image_input)
    else:
        img = image_input
 
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    score = float(model.predict(np.expand_dims(img, 0), verbose=0)[0][0])
 
    return {
        'stress_score' : round(score, 4),
        'prediction'   : 'STRESS' if score >= 0.5 else 'NO-STRESS',
        'confidence'   : round(abs(score - 0.5) * 2, 4),
    }
 
 
# Test it
sample_path = list((RAF_TEST_DIR / '2').glob('*.jpg'))[0]
result      = predict_stress(str(sample_path))
print(f"\n🧪  Test prediction on a Fear image:")
print(f"    Score      : {result['stress_score']}")
print(f"    Prediction : {result['prediction']}")
print(f"    Confidence : {result['confidence']}")
 
print(f"""
{'='*60}
  ✅  ALL DONE!
 
  AUC achieved : {rauc:.4f}
  Files saved:
    stress_model_final.keras
    eda_distribution.png
    training_curves.png
    evaluation.png
    emotion_stress_scores.png  ← KEY CHART FOR TEACHER
    gradcam_results.png
 
  What to show teacher:
    1. eda_distribution.png    → data understanding
    2. emotion_stress_scores.png → model understands which
                                   emotions = stress and why
    3. evaluation.png          → how accurate it is
    4. gradcam_results.png     → WHERE it looks on the face
{'='*60}
""")
 


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL [14]  —  TESTING PIPELINE WITH VISUALS             ║
# ╚══════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt

def test_and_visualize(image_path):
    """
    Takes an image path, predicts stress, and plots the 
    original image alongside its Grad-CAM heatmap.
    """
    print(f"🔍 Testing Image: {image_path.split('/')[-1]}")
    
    # 1. Run the prediction
    result = predict_stress(image_path)
    
    # 2. Load and prep image for visualization
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("❌ Error: Could not load image. Check the path.")
        return
        
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_arr = img_res.astype(np.float32) / 255.0

    # 3. Generate Grad-CAM
    heatmap = compute_gradcam(model, img_arr)
    jet     = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    jet_rgb = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(jet_rgb, 0.45, img_res, 0.55, 0)

    # 4. Plotting
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    color = '#e74c3c' if result['prediction'] == 'STRESS' else '#27ae60'
    
    axes[0].imshow(img_res)
    axes[0].set_title("Original Image", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(overlay)
    axes[1].set_title(f"Grad-CAM\nPred: {result['prediction']} ({result['stress_score']:.2f})", 
                      color=color, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Results:")
    print(f"   Prediction : {result['prediction']}")
    print(f"   Score      : {result['stress_score']}")
    print(f"   Confidence : {result['confidence'] * 100:.1f}%\n")

# --- Run Tests ---

# Test 1: Random Fear Image (Should be STRESS)
fear_folder = RAF_TEST_DIR / '2'
if fear_folder.exists():
    sample_fear = list(fear_folder.glob('*.jpg'))[random.randint(0, 10)]
    test_and_visualize(str(sample_fear))

# Test 2: Random Happy Image (Should be NO-STRESS)
happy_folder = RAF_TEST_DIR / '4'
if happy_folder.exists():
    sample_happy = list(happy_folder.glob('*.jpg'))[random.randint(0, 10)]
    test_and_visualize(str(sample_happy))

# Test 3: Custom Image (Uncomment and change path to test your own images)
# custom_image_path = "/kaggle/input/your-custom-image-folder/my_face.jpg"
# test_and_visualize(custom_image_path)